In [8]:
#transformer总体架构
#输入：文本序列
#输出：文本序列
#编码器部分
#解码器部分

#输入部分包含词嵌入层和位置编码层
#词嵌入层将输入的文本序列转换为向量表示
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系

#编码器部分包含多个编码器层，每个编码器层由多头自注意力机制和前馈神经网络组成
#每个编码器层由两个子层组成：多头自注意力机制和前馈神经网络
#第一个子层是多头自注意力机制以及一个残差连接，它允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系，残差连接有助于缓解深层网络中的梯度消失问题
#第二个子层包括一个前馈神经网络和规范化层以及一个残差连接，前馈神经网络由两个线性变换和一个激活函数组成，规范化层用于稳定训练过程，残差连接有助于缓解深层网络中的梯度消失问题

#解码器部分也包含多个解码器层，每个编码器层由三个子层组成：多头自注意力机制、编码器-解码器注意力机制和前馈神经网络，第一个子层为带掩码的多头自注意力机制以及残差连接和规范化层，第二个子层为多头自注意力机制层以及残差连接和规范化层，第三个子层为前馈神经网络层以及残差连接和规范化层

In [9]:
#文本嵌入层
#文本嵌入层将输入的文本序列转换为向量表示
#注意：在文本嵌入时，经常在embedding后乘以一个缩放因子，通常是嵌入维度的平方根。这是因为在Transformer中，输入的嵌入向量会被送入多头自注意力机制中进行计算，而多头自注意力机制中的点积操作可能会导致数值不稳定。通过乘以缩放因子，可以使得输入的嵌入向量的数值范围更适合进行点积计算，从而提高模型的训练稳定性和性能。

#文本嵌入层实现
import torch.nn as nn
import torch
import math


from transformers import BertTokenizer

embedding=nn.Embedding(100,10,padding_idx=0) #词表大小为100，嵌入维度为10，padding_idx=0表示将索引为0的词视为填充词，不进行训练
input=torch.tensor([[1,2,3,4,0],[5,6,7,8,0]],dtype=torch.long) #输入的文本序列，包含两个样本，每个样本包含5个词，其中索引为0的词表示填充词
output=embedding(input)
print(output)

class Embedding(nn.Module):
    def __init__(self,vocab_size,embedding_dim):
        super().__init__()
        self.vocab_size=vocab_size
        self.embedding_dim=embedding_dim
        self.embed=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
    def forward(self,x):
        x=self.embed(x)
        return x*math.sqrt(self.embedding_dim) #乘以缩放因子

embedding=Embedding(100,20)
embedding_output=embedding(input)
print(embedding_output)

tensor([[[-1.3619e+00,  3.2969e+00, -1.3191e+00,  6.5532e-01,  1.0986e+00,
          -3.8144e-01,  9.0072e-01, -2.0177e+00, -1.1041e+00, -4.3906e-01],
         [ 3.4663e-01,  3.0548e-01, -3.2096e-01,  7.2894e-01, -1.2532e-01,
           1.2334e+00, -1.2961e+00,  1.3757e+00,  1.6450e+00,  6.9128e-02],
         [-5.0725e-01,  6.2968e-01,  2.0711e+00, -6.8457e-01, -1.2899e-01,
          -1.2467e+00,  1.0016e+00,  7.0547e-01, -1.1001e+00, -4.2966e-02],
         [-4.6517e-01, -6.9812e-01, -2.0899e+00,  7.4936e-01, -2.0411e-01,
          -5.8009e-01, -1.2221e+00, -1.0678e-01,  2.4764e-01, -4.6280e-01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[-2.0132e-01, -8.9127e-02,  2.4340e-01, -6.7220e-01, -2.3934e+00,
          -5.2282e-01, -2.9911e-01,  3.1461e+00,  1.5470e+00,  9.1843e-01],
         [ 6.0495e-01,  1.4080e+00,  1.5805e-01, -6.0876e-01,  1.7492e+00,
           1.2608

In [34]:
#位置编码层
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系，将索引转化为位置编码向量
#三角函数位置编码公式
#PE(pos,2i)=sin(pos/10000^(2i/d_model)) 奇数位使用正弦函数，偶数位使用余弦函数
#PE(pos,2i+1)=cos(pos/10000^(2i/d_model))
#其中pos表示单词在序列中的位置，i表示嵌入维度的索引，d_model表示嵌入维度的大小

#位置编码层的实现
class PositionalEncoding(nn.Module):
    def __init__(self,dropout,d_model,max_len):
        super().__init__()
        self.dropout=nn.Dropout(dropout) #dropout层用于防止过拟合,这里为什么放在位置编码层中？因为位置编码是输入到模型中的一个重要特征，通过在位置编码层中添加dropout，可以随机失活一些位置编码的维度，从而增强模型的泛化能力，防止模型过拟合训练数据，提高模型在测试数据上的表现。

        pe=torch.zeros(max_len,d_model) #初始化pe矩阵，大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小
        position=torch.arange(0,max_len,dtype=torch.long).unsqueeze(1) #生成位置索引，大小为(max_len, 1)，其中max_len表示最大序列长度
        div_term=torch.exp(torch.arange(0,d_model,2,dtype=torch.long)*-(math.log(10000)/d_model)) #计算分母项，大小为(d_model/2)，其中d_model表示嵌入维度的大小

        #position和div_term相乘得到一个大小为(max_len, d_model/2)的矩阵，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

         #计算位置编码矩阵，使用三角函数公式，词向量奇数位使用正弦函数，偶数位使用余弦函数
         #pe矩阵的大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        pe=pe.unsqueeze(0) #添加维度，与词嵌入矩阵维度匹配，大小为(1, max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        self.register_buffer('pe',pe) #将pe矩阵注册为模型的缓冲区，使其在模型保存和加载时能够正确处理，并且在训练过程中不会被更新
    def forward(self,x):
        x=x+self.pe[:,:x.size(1)]
        return self.dropout(x)

#测试位置编码层
position_encoding=PositionalEncoding(dropout=0.2,d_model=20,max_len=10)
position_x=position_encoding(embedding_output)
print(position_x)
print(position_x.shape)
print(embedding_output.size(1))

tensor([[[-3.3406e+00, -0.0000e+00,  9.7886e+00, -1.0779e+01, -1.5136e+00,
          -3.3567e+00,  3.1691e+00, -5.3657e+00,  9.6504e-01,  2.4686e+00,
           5.5847e-02, -1.2385e+01,  1.3721e+01,  5.0706e+00,  0.0000e+00,
          -1.5773e+00,  2.3825e+00, -8.2622e+00, -5.3861e+00, -5.0933e+00],
         [-1.1673e+01, -6.2311e+00, -1.0232e+00, -7.0056e+00,  2.4562e+00,
           7.7259e+00, -4.4579e+00,  9.8140e-01, -7.7582e+00,  1.5147e-01,
          -8.4199e+00,  2.0721e+00, -1.8100e+00,  8.0317e+00,  1.3560e+01,
          -1.7324e+00, -1.9898e+00,  6.1958e+00,  3.5790e-01, -1.8376e+00],
         [ 1.3529e+01, -6.9028e+00,  9.3537e+00,  5.6039e+00,  0.0000e+00,
           0.0000e+00, -0.0000e+00, -9.6647e+00,  0.0000e+00,  1.6340e+00,
           4.1980e+00,  0.0000e+00, -6.5771e+00,  3.8689e+00, -2.9585e+00,
          -5.0864e+00, -2.1884e+00,  0.0000e+00,  3.7791e+00,  3.6309e-01],
         [ 7.3415e+00, -8.4293e+00,  0.0000e+00, -6.1471e+00,  5.0203e+00,
           0.0000e+00,

In [11]:
#掩码张量的介绍
#掩码张量是一个布尔类型的张量，用于指示模型在计算注意力权重时应该忽略哪些位置。掩码张量的形状通常为(batch_size, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度。掩码张量中的值为True表示对应位置应该被忽略，值为False表示对应位置应该被考虑。
#在Transformer模型中，掩码张量主要用于两种情况：填充掩码和未来掩码。填充掩码用于指示模型在计算注意力权重时应该忽略填充位置，未来掩码用于指示模型在计算注意力权重时应该忽略未来位置，以防止模型在训练过程中看到未来的信息。

#transformer中自注意力机制的实现
#自注意力机制允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系。自注意力机制的核心是计算注意力权重，这些权重表示输入序列中每个位置对其他位置的关注程度。注意力权重的计算通常使用点积操作，具体来说，给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V，然后计算注意力权重矩阵A，A=softmax(QK^T/sqrt(d_k))，其中d_k是键矩阵的维度。最后，通过将注意力权重矩阵A与值矩阵V相乘，得到自注意力机制的输出矩阵O，O=AV。自注意力机制的输出矩阵O包含了输入序列中每个位置对其他位置的关注信息，从而使模型能够捕捉到输入序列中的长距离依赖关系。

#自注意力机制中的Q,K,V矩阵的计算
#给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。具体来说，给定一个输入序列的表示矩阵X，首先通过三个不同的线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。线性变换的参数通常是可学习的权重矩阵W_Q、W_K和W_V，分别用于计算查询矩阵Q、键矩阵K和值矩阵V。

In [12]:
#生成掩码向量（未来掩码）

#测试生成上三角矩阵
import numpy as np
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=1)) #生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=0)) #生成一个上三角矩阵，k=0表示主对角线以上的元素保留，主对角线以下的元素为0

size=5
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int)) #生成一个掩码举着呢，三维，np.triu生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0，np.ones((1,size,size))生成一个全1的三维矩阵，1-np.triu(...)将上三角矩阵中的1变为0，0变为1，最后使用torch.from_numpy将其转换为PyTorch张量
print(mask)

[[0 1 1]
 [0 0 1]
 [0 0 0]]
[[1 1 1]
 [0 1 1]
 [0 0 1]]
tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)


In [13]:
#sentence_mask：用于解码器中的未来掩码
#生成一个句子掩码，用于指示模型在计算注意力权重时应该忽略哪些位置。句子掩码通常用于指示模型在计算注意力权重时应该忽略填充位置，以防止模型在训练过程中看到填充的信息。

#例如，给定一个输入序列的表示矩阵X和一个对应的句子掩码mask，模型在计算注意力权重时会将mask中为True的位置对应的注意力权重设置为负无穷，从而使得这些位置在计算注意力权重时被忽略。


In [14]:
#sentence_mask演示
torch.manual_seed(42)
input=torch.randn(1,size,size)
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int))
print(mask)
print(input)
input.masked_fill_(mask==0,float('-inf')) #使用掩码填充的方法将mask中为0的位置赋值为负无穷，最后使得这些位置在计算权重的时候被忽略

tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
         [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
         [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
         [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])


tensor([[[ 1.9269,    -inf,    -inf,    -inf,    -inf],
         [-1.2345, -0.0431,    -inf,    -inf,    -inf],
         [-0.4934,  0.2415, -1.1109,    -inf,    -inf],
         [-0.2168, -1.3847, -0.3957,  0.8034,    -inf],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])

In [15]:
#padding_mask的原理：用于编码器中的填充掩码
#生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置。填充掩码通常用于指示模型在计算注意力权重时应该忽略填充位置，以防止模型在训练过程中看到填充的信息。

#padding_mask的实现
a=torch.randn(size,size)
print(a!=0) #生成一个布尔类型的掩码张量，大小为(size, size)，其中size表示序列长度，a!=0表示将a中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False
a[a!=0]=1
print(a)

b=torch.tensor([[1,2,0,0],[4,5,0,0],[0,0,0,0],[0,0,0,0]])
padding_mask=(b!=0) #生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置，大小为(batch_size, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，b!=0表示将b中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False
print(padding_mask)


tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])
tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
tensor([[ True,  True, False, False],
        [ True,  True, False, False],
        [False, False, False, False],
        [False, False, False, False]])


In [16]:
#实现自注意力计算
import torch.nn.functional as F
#F表示可以直接调用里面的函数，不会学习参数
class Selfattention(nn.Module): #实现自注意力机制的计算，输入为查询矩阵Q、键矩阵K和值矩阵V，以及可选的掩码张量mask和dropout概率dropout
    def __init__(self,q,k,v,mask=None,dropout=None): #q=k=v=position_x,自注意力机制
        super().__init__()
        self.q=q
        self.k=k
        self.v=v
        self.mask=mask
        self.dropout=dropout
    def forward(self):
        d_model=self.q.size(-1) #词嵌入维度
        scores=torch.matmul(self.q,self.k.traspose(-2,-1))/torch.sqrt(torch.tensor(d_model))  #计算注意力权重矩阵，使用点积操作，大小为(batch_size, seq_len, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，self.q.size(-1)表示查询矩阵的最后一个维度，即词嵌入维度
        weights=F.softmax(scores,dim=-1) #进行归一化，得到注意力权重矩阵，大小为(batch_size, seq_len, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，dim=-1表示在最后一个维度上进行softmax操作，即对每个查询位置的注意力权重进行归一化，使得它们的和为1

        if mask is not None:
            weights=weights.masked_fill_(mask==0,float('-inf')) #掩码操作，将mask中为0的位置对应的注意力权重设置为负无穷，最后使得这些位置在计算权重的时候被忽略

        if self.dropout is not None:
            weights=nn.Dropout(self.dropout)(weights)

        return torch.matmul(weights,self.v),weights


In [17]:
#多头注意力机制的理解
#多头注意力机制是Transformer模型中的一个重要组件，它允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系。多头注意力机制通过将查询矩阵Q、键矩阵K和值矩阵V分别映射到多个不同的子空间中，来实现对输入序列的多方面关注。具体来说，给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V，然后将这些矩阵分别划分为多个头，每个头对应一个不同的子空间。对于每个头，计算注意力权重矩阵A，并将其与值矩阵V相乘，得到每个头的输出矩阵O。最后，将所有头的输出矩阵O进行拼接，并通过线性变换得到多头注意力机制的最终输出矩阵。这种机制使得模型能够同时关注输入序列中的多个位置，从而更好地捕捉到输入序列中的长距离依赖关系。

#例如输入注意力计算之前，q=k=v=[2,4,512]，多头注意力机制将其映射到多个不同的子空间中，例如8个头，每个头对应一个64维的子空间，那么每个头的查询矩阵Q、键矩阵K和值矩阵V的大小分别为[2,4,64]。对于每个头，计算注意力权重矩阵A，并将其与值矩阵V相乘，得到每个头的输出矩阵O，大小为[2,4,64]。最后，将所有头的输出矩阵O进行拼接，并通过线性变换得到多头注意力机制的最终输出矩阵，大小为[2,4,512]。

In [18]:
#多头注意力机制的实现
class MultiheadAttention(nn.Module):
    def __init__(self,embedding_dim,heads,dropout):
        super().__init__()
        self.embedding_dim=embedding_dim
    
        
        self.heads=heads
        self.dropout=dropout

        assert embedding_dim%heads==0 #确保嵌入维度能够被头数整除
        #assert语句用于检查条件是否满足，如果条件不满足，则会抛出一个AssertionError异常，并显示指定的错误消息。在这里，assert语句用于确保嵌入维度能够被头数整除，以便每个头的维度能够正确计算。

        self.d_k=embedding_dim//heads #每个头的维度

        self.q_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到查询矩阵Q
        self.k_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到键矩阵K
        self.v_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将输入的表示矩阵映射到值矩阵V

        self.out_linear=nn.Linear(embedding_dim,embedding_dim) #线性变换，用于将多头注意力机制的输出矩阵映射回原始维度

        self.attention_weights=None
        self.dropout=nn.Dropout(dropout)
    def forward(self,q,k,v,mask=None):

        batch_size=q.size(0)
        #进行线性投影
        q=self.q_linear(q)
        k=self.k_linear(k)
        v=self.v_linear(v)

        #进行多头切分
        q=q.view(batch_size,-1,self.heads,self.d_k)
        k=k.view(batch_size,-1,self.heads,self.d_k)
        v=v.view(batch_size,-1,self.heads,self.d_k) 

        #进行维度交换
        q=q.transpose(1,2)
        k=k.transpose(1,2)
        v=v.transpose(1,2)

        #进行自注意力计算
        scores=torch.matmul(q,k.transpose(-2,-1))/torch.sqrt(torch.tensor(self.d_k))

        #掩码处理
        if mask is not None:
            scores.masked_fill_(mask==0,float('-inf'))
        
        weights=F.softmax(scores,dim=-1)

        weights=self.dropout(weights)  #这里为什么需要dropout？因为在训练过程中，dropout可以随机失活一些注意力权重，从而增强模型的泛化能力，防止模型过拟合训练数据，提高模型在测试数据上的表现。通过在计算注意力权重之后应用dropout，可以使得模型在训练过程中更具鲁棒性，并且能够更好地适应不同的输入数据。

        output=torch.matmul(weights,v) #进行注意力计算，得到多头注意力的初始输出矩阵，大小为(batch_size, heads, seq_len, d_k)，其中batch_size表示批次大小，heads表示头数，seq_len表示序列长度，d_k表示每个头的维度

        output=output.transpose(1,2) #将heads和seq_len进行维度交换，得到大小为(batch_size, seq_len, heads, d_k)的矩阵，其中batch_size表示批次大小，seq_len表示序列长度，heads表示头数，d_k表示每个头的维度
        output=output.contiguous().view(batch_size,-1,self.embedding_dim) #将最后两个维度相乘，得到大小为(batch_size, seq_len, embedding_dim)的矩阵，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小

        output=self.out_linear(output) #进行输出维度的映射
        self.attention_weights=weights
        return output,weights

#测试
multihead=MultiheadAttention(embedding_dim=512,heads=8,dropout=0.1)
q=torch.randn(2,4,512)
k=q
v=q
mask=torch.from_numpy(1-np.triu(np.ones((2,4,4)),k=1).astype(int))
mask=mask.unsqueeze(1) #添加一个维度，使其与多头注意力机制的输入维度匹配，大小为(2, 1, 4, 4)，其中2表示批次大小，1表示头数，4表示序列长度
print(mask)
multi_output,weights=multihead(q,k,v,mask)
print(multi_output.shape)

print(weights.shape)

tensor([[[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]],


        [[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]]], dtype=torch.int32)
torch.Size([2, 4, 512])
torch.Size([2, 8, 4, 4])


In [19]:
#前馈全连接层
#前馈全连接层是Transformer模型中的一个重要组件，它由两个线性变换和一个激活函数组成。前馈全连接层的输入是来自多头注意力机制的输出矩阵，大小为(batch_size, seq_len, embedding_dim)，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小。前馈全连接层首先通过一个线性变换将输入矩阵映射到一个更高维度的空间中，通常是4倍的嵌入维度，即hidden_dim=4*embedding_dim。然后，通过一个激活函数（例如ReLU）对映射后的矩阵进行非线性变换，得到一个新的矩阵。最后，通过另一个线性变换将这个新的矩阵映射回原始的嵌入维度，得到前馈全连接层的输出矩阵，大小为(batch_size, seq_len, embedding_dim)。

# 前馈全连接层的作用是对多头注意力机制的输出进行进一步的处理和变换，以增强模型的表达能力和非线性特征。

class Feedforward(nn.Module):
    def __init__(self,d_model,d_ff,dropout):
        super().__init__()
        self.d_model=d_model #词向量维数
        self.d_ff=d_ff #全连接层的维数，通常是d_model的四倍
        self.dropout=nn.Dropout(dropout) #dropout用于随机失活
        self.fc1=nn.Linear(self.d_model,self.d_ff)
        self.fc2=nn.Linear(self.d_ff,self.d_model) #最终结果又映射成原始的维度
    def forward(self,x):
        x=self.fc1(x)
        x=F.relu(x) #激活函数
        x=self.dropout(x)
        x=self.fc2(x)
        return x

#测试
feedback=Feedforward(d_model=512,d_ff=2048,dropout=0.1)
input=multi_output
feed=feedback(input)
print(feed)

print(feed.shape)



tensor([[[ 0.0043, -0.0082, -0.0563,  ..., -0.0943,  0.0725,  0.0785],
         [ 0.1360,  0.0641, -0.0597,  ..., -0.0512, -0.0270,  0.0379],
         [-0.0072,  0.0947,  0.0012,  ...,  0.0483,  0.0097, -0.0164],
         [ 0.0362,  0.0506, -0.0196,  ...,  0.0067,  0.0337,  0.0340]],

        [[ 0.1485,  0.0874, -0.0427,  ..., -0.0175, -0.0225,  0.1304],
         [-0.0373,  0.0954, -0.0694,  ...,  0.0330, -0.0096,  0.0566],
         [ 0.0164,  0.0583, -0.1032,  ...,  0.0098, -0.0561,  0.0890],
         [ 0.0720,  0.0175, -0.0874,  ...,  0.0006, -0.0015,  0.0348]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 4, 512])


In [20]:
#规范化层
#规范化层是Transformer模型中的一个重要组件，它用于稳定训练过程并加速模型的收敛。规范化层通过对输入数据进行归一化处理，使得输入数据的分布更加稳定，从而有助于缓解梯度消失和梯度爆炸问题。规范化层通常使用Layer Normalization（层归一化）或Batch Normalization（批归一化）来实现。

#Layernormalization是对每个样本的特征进行归一化处理，计算每个样本的均值和标准差，并使用这些统计量来对输入数据进行归一化。Layer Normalization在Transformer模型中更常用，因为它不依赖于批次大小，可以更好地适应不同的输入序列长度。

class Layernormal(nn.Module):
    def __init__(self,features,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.weight=nn.Parameter(torch.ones(features)) #可训练的参数，对归一化之后的数据进行缩放，作为权重使用
        self.bias=nn.Parameter(torch.zeros(features)) #可训练的参数，对归一化的数据进行平移，作为偏置使用
    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True) #求解平均值
        var=x.var(dim=-1,keepdim=True) #求解方差
        x=(x-mean)/torch.sqrt(var+self.eps) #归一化处理
        return self.weight*x+self.bias #对归一化之后的数据进行缩放和平移，得到输出，这样是为了避免归一化之后的数据过于平坦，失去表达能力，通过引入可训练的权重和偏置，可以让模型在训练过程中学习到适合当前任务的缩放和平移参数，从而增强模型的表达能力。

#测试
layernorm=Layernormal(512)
input=feed
norm_output=layernorm(input)
print(norm_output)

tensor([[[ 0.0022, -0.1299, -0.6392,  ..., -1.0427,  0.7249,  0.7888],
         [ 2.0833,  0.9426, -1.0206,  ..., -0.8862, -0.5015,  0.5280],
         [-0.1410,  1.9727,  0.0333,  ...,  1.0112,  0.2097, -0.3311],
         [ 0.7802,  1.0948, -0.4408,  ...,  0.1340,  0.7252,  0.7320]],

        [[ 1.7316,  1.0042, -0.5441,  ..., -0.2444, -0.3045,  1.5162],
         [-0.6048,  1.4213, -1.0950,  ...,  0.4693, -0.1821,  0.8290],
         [ 0.2780,  1.0435, -1.9101,  ...,  0.1568, -1.0489,  1.6047],
         [ 1.5760,  0.3586, -1.9836,  ..., -0.0198, -0.0654,  0.7446]]],
       grad_fn=<AddBackward0>)


In [21]:
#子层连接结构的实现

#子层连接结构的作用：
#子层连接结构是Transformer模型中的一个重要组件，它用于连接多头注意力机制和前馈全连接层等子层，并通过残差连接和规范化层来稳定训练过程。子层连接结构的输入是来自前一个子层的输出矩阵，大小为(batch_size, seq_len, embedding_dim)，其中batch_size表示批次大小，seq_len表示序列长度，embedding_dim表示嵌入维度的大小。子层连接结构首先将输入矩阵传递给当前子层进行处理，得到一个新的矩阵。然后，通过一个残差连接将这个新的矩阵与输入矩阵相加，得到一个新的矩阵。最后，通过一个规范化层对这个新的矩阵进行归一化处理，得到子层连接结构的输出矩阵，大小为(batch_size, seq_len, embedding_dim)。
class Sublayerconnection(nn.Module):
    def __init__(self,size,dropout):
        super().__init__()
        self.norm=Layernormal(size,dropout)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x,sublayer):
        # return x+self.dropout(self.norm(sublayer(x))) #post_norm结构，先进行子层处理，再进行规范化和dropout，最后与输入矩阵x相加得到输出矩阵

        # pre_norm结构，先进行规范化和子层处理，再dropout，最后与输入矩阵x相加得到输出矩阵,更稳定的训练过程，适合深层网络
        return x+self.dropout((sublayer(self.norm(x))))

        #sublayer是一个函数，表示当前子层的处理逻辑，例如多头注意力机制或前馈全连接层，sublayer(x)表示将输入矩阵x传递给当前子层进行处理，得到一个新的矩阵，然后通过规范化层进行归一化处理，并通过dropout进行随机失活，最后将这个新的矩阵与输入矩阵x相加，得到子层连接结构的输出矩阵。

#使用了残差连接的思想，即将子层的输出与输入矩阵相加，形成一个新的矩阵，这样可以帮助缓解深层网络中的梯度消失问题，使得模型能够更好地学习到输入数据的特征。同时，通过规范化层和dropout来稳定训练过程，增强模型的泛化能力。

#测试子层连接结构
sublayer_connection=Sublayerconnection(20,0.1) #定义子层连接结构，size=20表示嵌入维度的大小，dropout=0.1表示dropout的概率
mutiheadlayer=MultiheadAttention(20,4,0.1) #用子层连接结构包裹多头注意力机制层

sublayer=lambda x:mutiheadlayer(x,x,x,mask=None)[0] #定义匿名函数，返回的是多头注意力的输出矩阵，注意多头注意力机制的类有两个输出，第一个是输出矩阵，第二个是注意力权重矩阵，这里我们只需要输出矩阵，所以使用[0]来获取多头注意力机制的输出矩阵

sublayer_output=sublayer_connection(position_x,sublayer) #输入带位置编码的矩阵与包裹的匿名函数
print(sublayer_output)




tensor([[[-4.3479e+00, -2.0093e+00,  9.6825e+00, -1.3318e+01, -3.5310e-01,
          -5.2362e+00,  4.0532e+00, -0.0000e+00,  1.9750e+00,  2.0845e+00,
           5.3527e-01, -1.2795e+01,  1.4086e+01,  5.5313e+00,  6.6958e+00,
          -1.5880e+00,  2.5390e+00, -8.2931e+00, -5.4624e+00, -5.0212e+00],
         [-1.3736e+01, -6.5950e+00, -1.6670e+00, -9.2987e+00,  9.4693e-02,
          -8.1580e-02, -1.5517e-01,  1.2099e-01,  1.0418e-01, -2.7764e-01,
          -8.0759e+00,  1.7442e+00,  7.0113e-02,  8.4771e+00,  1.3775e+01,
          -1.8129e+00, -1.6984e+00,  6.0003e+00,  2.1801e-01, -1.6388e+00],
         [ 1.1393e+01, -6.1415e+00,  8.3543e+00,  3.5188e+00,  4.8331e+00,
          -7.8125e-02, -5.2536e+00, -1.0674e+01,  5.8689e+00,  1.1637e+00,
           4.5873e+00,  3.9429e+00, -6.1992e+00,  4.3894e-01, -2.8047e+00,
          -5.1251e+00, -1.8322e+00,  1.6894e+00,  3.6782e+00,  1.1563e-01],
         [ 6.1167e+00, -6.8165e-02,  5.4918e-01,  5.4710e-02,  5.6010e+00,
           9.0629e+00,

In [22]:
#编码器层的作用
#每个编码器层完成一次对输入的特征提取过程，即编码过程
#实现编码器类
class Encoderlayer(nn.Module):
    def __init__(self,d_model,d_ff,heads,dropout):
        super().__init__()
        self.d_model=d_model
        self.heads=heads
        self.dropout=dropout
        self.attention=MultiheadAttention(d_model,heads,dropout) #多头注意力机制
        self.feedforward=Feedforward(d_model,d_ff,dropout) #前馈全连接层
        self.sublayer1=Sublayerconnection(d_model,dropout) #子层连接结构，用于连接注意力层
        self.sublayer2=Sublayerconnection(d_model,dropout) #子层连接结构，用于连接前馈全连接层
    def forward(self,x,mask=None):
        x=self.sublayer1(x,lambda x:self.attention(x,x,x,mask)[0])
        x=self.sublayer2(x,lambda x:self.feedforward(x))
        return x

#测试编码器层
encoder_layer=Encoderlayer(20,80,5,0.2)
input=position_x
print(position_x.shape)
mask=torch.from_numpy(1-np.triu(np.ones((2,5,5)),k=1).astype(int)).unsqueeze(1) #生成一个掩码张量，大小为(2, 1, 5, 5)，其中2表示批次大小，1表示头数，5表示序列长度，np.triu生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0，np.ones((2,5,5))生成一个全1的四维矩阵，1-np.triu(...)将上三角矩阵中的1变为0，0变为1，最后使用torch.from_numpy将其转换为PyTorch张量，并通过unsqueeze(1)添加一个维度，使其与多头注意力机制的输入维度匹配，然后通过广播机制将其扩展为(2, 4, 5, 5)，其中4表示头数，5表示序列长度，最后得到的掩码张量可以用于多头注意力机制的计算，指示模型在计算注意力权重时应该忽略哪些位置。
print(mask)
encoder_output=encoder_layer(position_x,mask)
print(encoder_output)
print(encoder_output.shape)


torch.Size([2, 5, 20])
tensor([[[[1, 0, 0, 0, 0],
          [1, 1, 0, 0, 0],
          [1, 1, 1, 0, 0],
          [1, 1, 1, 1, 0],
          [1, 1, 1, 1, 1]]],


        [[[1, 0, 0, 0, 0],
          [1, 1, 0, 0, 0],
          [1, 1, 1, 0, 0],
          [1, 1, 1, 1, 0],
          [1, 1, 1, 1, 1]]]], dtype=torch.int32)
tensor([[[ -5.5313,  -0.1327,   9.7721, -13.3893,   0.1341,  -6.8193,   4.0540,
            1.7502,   1.8551,   1.5993,   0.1420, -12.1847,  12.3573,   5.5256,
            7.1773,  -1.7910,   2.2064,  -7.5586,  -4.7603,  -4.5313],
         [-13.9232,  -5.5321,  -1.7083,  -9.2195,  -0.0677,  -1.2725,   0.1843,
            0.4059,   0.1935,  -0.6222,  -7.9668,   1.7552,  -1.1254,   8.1814,
           14.1320,  -1.4721,  -1.7151,   6.3380,   0.4358,  -1.5347],
         [ 10.7050,  -5.5240,   8.3829,   2.9211,   4.5883,  -0.0385,  -5.2076,
          -10.6784,   5.6501,   1.1446,   5.1426,   4.3312,  -5.6799,   0.1192,
           -3.3632,  -4.7415,  -2.2577,   2.7531,   3.7551,

In [23]:
#堆叠编码器层，实现完整的编码器结构
#编码器结构
#编码器由多个编码器层堆叠而成，每个编码器层完成一次对输入的特征提取过程，即编码过程。通过堆叠多个编码器层，模型能够逐步提取输入序列中的高级特征，从而更好地理解输入序列的语义信息。每个编码器层都包含多头注意力机制和前馈全连接层，通过子层连接结构将它们连接起来，以稳定训练过程并增强模型的表达能力。
class Encoder(nn.Module):
    def __init__(self,d_model,d_ff,heads,dropout,layer_num):
        super().__init__()
        self.layers=nn.ModuleList() #使用modulelist存储多个编码器层
        for i in range(layer_num):
            layer=Encoderlayer(d_model,d_ff,heads,dropout)
            self.layers.append(layer) #添加编码器层至列表中
    def forward(self,x,mask=None):
        for layer in self.layers:
            x=layer(x,mask)
        return x

#测试
encoder=Encoder(20,80,5,0.2,4)
encoder_input=position_x
encoder_output=encoder(encoder_input,mask)
print(encoder_output)
print(encoder_output.shape)


tensor([[[-3.3274e+00, -1.1742e+00,  9.2777e+00, -1.4851e+01,  3.3417e-03,
          -5.2451e+00,  5.6451e+00, -4.9488e-01,  1.2932e+00,  1.8147e+00,
          -1.5891e-01, -1.1870e+01,  1.6427e+01,  3.6376e+00,  6.5407e+00,
          -1.9341e+00,  3.2344e+00, -6.7147e+00, -4.5380e+00, -5.3372e+00],
         [-1.5256e+01, -4.1549e+00, -2.1834e+00, -1.0470e+01,  8.4714e-01,
          -8.7444e-01,  1.1859e+00, -4.5024e-01, -1.2469e+00, -3.1901e-01,
          -8.7248e+00,  1.6879e+00, -2.8929e-01,  5.9514e+00,  1.4086e+01,
          -1.3012e+00, -1.8163e+00,  5.8800e+00,  2.7378e-02, -1.9218e+00],
         [ 1.0765e+01, -4.1273e+00,  8.6499e+00,  3.0871e+00,  5.9587e+00,
          -1.1942e+00, -3.8489e+00, -1.2161e+01,  5.2055e+00,  1.2413e+00,
           4.6481e+00,  3.6267e+00, -6.2356e+00, -2.9488e-01, -2.3639e+00,
          -4.6282e+00, -1.2905e+00,  8.0339e-01,  3.0357e+00, -6.5580e-02],
         [ 5.6439e+00,  1.0181e+00, -2.1609e-01, -1.2655e+00,  6.3189e+00,
           7.7215e+00,

In [24]:
#解码器层的实现
#解码器层的结构
#每个解码器层由三个子层组成：多头自注意力机制、编码器-解码器注意力机制和前馈神经网络，第一个子层为带掩码的多头自注意力机制以及残差连接和规范化层，第二个子层为多头自注意力机制层以及残差连接和规范化层，第三个子层为前馈神经网络层以及残差连接和规范化层
class Decoderlayer(nn.Module):
    def __init__(self,d_model,d_ff,heads,dropout):
        super().__init__()
        self.d_model=d_model
        self.heads=heads
        self.dropout=nn.Dropout(dropout) #dropout层的位置通常放在子层连接结构中，用于对每个子层的输出进行随机失活，以防止过拟合并增强模型的泛化能力
        self.Multiattention=MultiheadAttention(d_model,heads,dropout)
        self.attention=MultiheadAttention(d_model,heads,dropout)
        self.feedforward=Feedforward(d_model,d_ff,dropout)
        self.sublayer1=Sublayerconnection(d_model,dropout)
        self.sublayer2=Sublayerconnection(d_model,dropout)
        self.sublayer3=Sublayerconnection(d_model,dropout)

    def forward(self,x,encoder_output,src_mask=None,tar_mask=None):
        x=self.sublayer1(x,lambda x:self.Multiattention(x,x,x,tar_mask)[0])
        x=self.sublayer2(x, lambda x:self.attention(x,encoder_output,encoder_output,src_mask)[0])
        x=self.sublayer3(x, lambda x:self.feedforward(x))
        return x
#测试
decoder_layer=Decoderlayer(20,80,5,0.2)
decoder_input=position_x #解码器的输入通常是带有位置编码的目标序列的嵌入表示，或者是前一个解码器层的输出
decoder_output=decoder_layer(decoder_input,encoder_output,mask)
print(decoder_output)
print(decoder_output.shape)



tensor([[[-6.9106e+00, -1.1907e+00,  1.0311e+01, -1.3167e+01,  3.0544e+00,
          -4.7014e+00,  7.8039e+00,  2.6035e+00,  2.9554e+00,  2.8450e+00,
          -3.0706e+00, -1.4417e+01,  1.0570e+01,  5.3560e+00,  7.2156e+00,
          -6.9389e+00, -1.9925e+00, -1.1401e+01, -5.9837e+00, -8.8149e+00],
         [-1.2053e+01, -7.3731e+00, -1.5529e+00, -1.2178e+01,  3.6564e+00,
          -1.7310e-01,  3.2555e+00,  1.4988e-02,  0.0000e+00,  1.6831e+00,
          -7.8331e+00,  1.4841e+00, -3.2234e+00,  8.0811e+00,  9.3566e+00,
          -4.9118e+00, -3.5393e+00,  4.9799e+00, -4.1264e-01, -2.3482e+00],
         [ 7.7744e+00, -4.7994e+00,  8.3121e+00,  2.3260e+00,  6.2902e+00,
          -1.2743e+00, -3.8399e+00, -9.6982e+00,  4.1604e+00, -1.1803e+00,
           5.0531e+00,  3.7836e+00, -5.9255e+00, -1.2078e+00, -6.3927e+00,
          -8.5181e+00, -2.2563e+00,  1.4511e+00,  2.8945e+00, -8.7963e-02],
         [ 7.1239e+00,  1.6402e+00, -2.3130e+00,  9.8560e-01,  5.5896e+00,
           8.8505e+00,

In [25]:
#堆叠解码器层，实现完整的解码器结构
class Decoder(nn.Module):
    def __init__(self,d_model,d_ff,heads,dropout,layer_num):
        super().__init__()
        self.layers=nn.ModuleList()
        for i in range(layer_num):
            layer=Decoderlayer(d_model,d_ff,heads,dropout)
            self.layers.append(layer)

    def forward(self,x,encoder_output,src_mask=None,tar_mask=None):
        for layer in self.layers:
            x=layer(x,encoder_output,src_mask,tar_mask) #解码器的输入通常是带有位置编码的目标序列的嵌入表示，或者是前一个解码器层的输出，mask通常是未来掩码，用于指示模型在计算注意力权重时应该忽略未来位置，以防止模型在训练过程中看到未来的信息
        return x


#测试
decoder=Decoder(20,80,5,0.2,4)
decoder_input=position_x
decoder_output=decoder(decoder_input,encoder_output,mask)
print(decoder_output)
print(decoder_output.shape)

tensor([[[-1.2310e+01, -3.6233e+00,  2.4509e+01, -1.8174e+01, -1.0061e+01,
          -1.4825e+01,  5.5906e+00, -4.7654e+00,  2.5709e+00,  6.0020e-02,
           7.8209e+00, -1.3242e+01,  8.0502e+00, -6.8436e+00,  4.5323e+00,
           2.3385e+00,  3.1103e+00, -1.7285e+01,  6.1902e+00,  4.5904e+00],
         [-1.8306e+01, -9.6686e+00,  2.1796e+00, -9.4034e+00, -4.5775e+00,
          -6.5463e+00, -6.1741e-01, -2.9740e+00,  6.5369e+00, -5.9058e+00,
          -2.2749e+00, -6.5373e-01,  3.1749e+00, -4.4417e+00,  1.4838e+01,
           2.1716e-01, -9.6229e+00,  2.6144e+00,  6.9581e+00, -3.5111e+00],
         [ 1.2815e+00, -1.6106e+01,  5.7687e+00,  4.8026e-01,  5.9495e+00,
          -2.4531e+00, -8.7766e+00, -1.2051e+01,  3.0105e+00,  9.8132e+00,
          -3.0003e+00,  8.9438e+00, -4.6031e+00, -3.9635e+00, -5.8566e+00,
           2.5472e+00,  3.2181e+00,  6.0919e+00,  4.3471e+00,  4.2059e+00],
         [ 3.8116e+00,  2.3672e+00,  7.4490e+00, -6.9761e+00, -2.0114e-01,
           9.8282e+00,

In [26]:
#输出层实现
class Generator(nn.Module):
    def __init__(self,d_model,vocab_size):
        super().__init__()
        self.linear=nn.Linear(d_model,vocab_size)
    def forward(self,x):
        x=self.linear(x)
        x=F.log_softmax(x,dim=-1) #这里使用log_softmax是为了在训练过程中计算交叉熵损失时更加稳定，log_softmax将输出转换为对数概率，可以避免数值不稳定的问题，并且在计算交叉熵损失时可以直接使用负对数概率进行计算，从而提高模型的训练效率和性能
        return x

#测试
generator=Generator(20,100)
generator_input=decoder_output
generator_output=generator(generator_input)
print(generator_output)
print(generator_output.shape)

tensor([[[ -1.4212, -20.4526, -21.0408, -13.8001,  -9.7063, -11.8313,  -7.1131,
          -26.4953,  -3.0139, -15.9123, -15.0272, -11.4477,  -2.3649,  -7.8059,
          -14.5025,  -2.8303, -11.2655, -12.6245,  -6.5537,  -8.7248, -14.1217,
           -5.7327, -21.7145, -11.9537, -11.6642,  -5.1960, -10.3583, -17.8896,
           -5.4044, -10.4297, -12.7939, -16.9915, -12.7315, -22.7505, -17.4046,
           -6.8587,  -8.6374, -10.1551, -18.0530,  -6.2202,  -3.3331,  -8.6493,
           -8.2931,  -9.1702, -15.9835, -12.2050,  -5.7892, -20.4348, -11.2139,
           -5.8345,  -7.0340,  -8.3540, -11.7246, -10.0579,  -5.7613, -21.9152,
          -15.3467,  -5.6857, -14.1667,  -2.7717, -12.1709, -15.8660, -10.3821,
          -10.3544, -15.0174,  -6.6268,  -6.2772,  -9.6817, -10.5784, -11.1506,
          -13.6043,  -3.6198, -15.2870,  -9.5590,  -5.4706, -13.8185, -16.2141,
          -11.4871, -14.7619, -12.2632, -22.8435, -12.0182, -11.0190,  -9.5848,
           -2.8859, -11.1276,  -6.0858, 

In [37]:
#完整的Transformer模型实现
class Transformer(nn.Module):
    def __init__(self,d_model,src_embed,tar_embed,encoder,decoder,generator):
        super().__init__()
        self.d_model=d_model
        self.src_embed=src_embed
        self.tar_embed=tar_embed
        self.encoder=encoder
        self.decoder=decoder
        self.generator=generator
    def forward(self,x,y,source_mask1,source_mask2,target_mask):
        src_x=self.src_embed(x)
        tar_y=self.tar_embed(y)
        encoder_output=self.encoder(src_x,source_mask1) #source_mask1为填充掩码，source_mask2为填充掩码
        decoder_output=self.decoder(tar_y,encoder_output,source_mask2,target_mask)
        output=self.generator(decoder_output)
        return output

#测试和参数初始化
x=torch.randint(0,100,(2,4))
y=torch.randint(0,100,(2,6))

src_embed=nn.Sequential(Embedding(100,20),PositionalEncoding(0.1,20,10))
tar_embed=nn.Sequential(Embedding(100,20),PositionalEncoding(0.1,20,10))

encoder=Encoder(20,80,5,0.2,5)
decoder=Decoder(20,80,5,0.2,5)
generator=Generator(20,100)

transformer=Transformer(d_model=20,src_embed=src_embed,tar_embed=tar_embed,encoder=encoder,decoder=decoder,generator=generator)

source_mask1=torch.tensor(x!=0).unsqueeze(1).unsqueeze(1) #解码器的填充掩码
source_mask2=torch.tensor(x!=0).unsqueeze(1).unsqueeze(2) #编码器-解码器注意力掩码，是填充掩码

#掩码通过广播机制与多头注意力机制的输入维度匹配，source_mask1的大小为(batch_size, 1, 1, seq_len)，source_mask2的大小为(batch_size, 1, seq_len, 1)，其中batch_size表示批次大小，seq_len表示序列长度，x!=0表示将x中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False，通过unsqueeze(1)和unsqueeze(2)添加两个维度，使其与多头注意力机制的输入维度匹配，source_mask1用于指示模型在计算注意力权重时应该忽略哪些位置，source_mask2用于指示模型在计算注意力权重时应该忽略哪些位置，通常是填充位置，以防止模型在训练过程中看到填充的信息。


#生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置，大小为(batch_size, 1, 1, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，x!=0表示将x中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False，通过unsqueeze(1)和unsqueeze(2)添加两个维度，使其与多头注意力机制的输入维度匹配
tar_mask=torch.from_numpy(1-np.triu(np.ones((2,6,6)),k=1).astype(int)).unsqueeze(1)
output=transformer(x,y,source_mask1,source_mask2,tar_mask)
print(output)
print(output.shape)
print(source_mask1.shape)
print(source_mask2.shape)




tensor([[[-16.1973, -14.7040,  -9.6787,  ..., -14.2741, -13.6986, -11.1628],
         [-28.8149, -16.9386,  -5.5767,  ..., -13.1111, -24.2370,  -9.6535],
         [-14.2703, -13.6258, -11.6204,  ..., -13.5910, -20.4929, -10.5619],
         [ -8.4426, -11.5506,  -9.6862,  ..., -10.0760, -10.6162, -10.3942],
         [-10.6410,  -5.8168, -11.6146,  ...,  -8.3913, -11.3398,  -5.5554],
         [-15.7025, -13.8781,  -9.9783,  ..., -11.0073, -17.5722,  -9.2445]],

        [[ -1.4633, -11.3681,  -8.2156,  ..., -10.1752,  -4.2664,  -7.2737],
         [ -3.5421,  -6.4322,  -9.7711,  ...,  -9.2539,  -7.4947,  -7.1133],
         [ -8.1189,  -9.7472,  -6.7208,  ...,  -8.9126,  -9.4430, -13.6135],
         [ -8.7527, -10.2779,  -5.9598,  ...,  -9.5972,  -7.8986,  -9.5134],
         [-12.5830,  -6.7216,  -8.8105,  ...,  -6.2778, -11.7833, -11.3542],
         [-15.3907, -12.5339,  -9.2777,  ...,  -9.7816, -11.2145, -13.3856]]],
       grad_fn=<LogSoftmaxBackward0>)
torch.Size([2, 6, 100])
torch.Size

C:\Users\icyw\AppData\Local\Temp\ipykernel_6048\447923856.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  source_mask1=torch.tensor(x!=0).unsqueeze(1).unsqueeze(1) #解码器的填充掩码
C:\Users\icyw\AppData\Local\Temp\ipykernel_6048\447923856.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  source_mask2=torch.tensor(x!=0).unsqueeze(1).unsqueeze(2) #编码器-解码器注意力掩码，是填充掩码


In [40]:
text='我喜欢学习自然语言处理。'
target='我喜欢学习机器学习。'
tokenizer=BertTokenizer.from_pretrained('bert-base-chinese')
#使用BertTokenizer对输入文本进行分词和编码，返回一个包含输入文本的token id的张量，padding='max_length'表示将输入文本填充到最大长度，truncation=True表示如果输入文本超过最大长度则进行截断，max_length=20表示最大长度为20，tokenizer.vocab_size表示词汇表的大小
x_tokens=tokenizer(text,return_tensors='pt',padding='max_length',truncation=True,max_length=20) #return_tensors='pt'表示返回一个PyTorch张量，padding='max_length'表示将输入文本填充到最大长度，truncation=True表示如果输入文本超过最大长度则进行截断，max_length=20表示最大长度为20
y_tokens=tokenizer(target,return_tensors='pt',padding='max_length',truncation=True,max_length=20)
vocab_size=tokenizer.vocab_size

x_index=x_tokens['input_ids']
y_index=y_tokens['input_ids']
print(x_index)
print(y_index)
print(y_index.shape)
transformer=Transformer(d_model=512,src_embed=nn.Sequential(Embedding(vocab_size,512),PositionalEncoding(0.1,512,20)),tar_embed=nn.Sequential(Embedding(vocab_size,512),PositionalEncoding(0.1,512,20)),encoder=Encoder(512,2048,8,0.1,6),decoder=Decoder(512,2048,8,0.1,6),generator=Generator(512,vocab_size))


source_mask1=torch.tensor(x_index!=0).unsqueeze(1).unsqueeze(1) #生成一个填充掩码，用于指示模型在计算注意力权重时应该忽略哪些位置，大小为(batch_size, 1, 1, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度，x_index!=0表示将x_index中不等于0的位置对应的掩码值设置为True，等于0的位置对应的掩码值设置为False，通过unsqueeze(1)和unsqueeze(2)添加两个维度，使其与多头注意力机制的输入维度匹配
source_mask2=torch.tensor(x_index!=0).unsqueeze(1).unsqueeze(1)
tar_mask=torch.from_numpy(np.triu(np.ones((1,20,20)),k=1).astype(int)).unsqueeze(1)
output=transformer(x_index,y_index,source_mask1,source_mask2,tar_mask)
print(output)
print(output.shape)

tensor([[ 101, 2769, 1599, 3614, 2110,  739, 5632, 4197, 6427, 6241, 1905, 4415,
          511,  102,    0,    0,    0,    0,    0,    0]])
tensor([[ 101, 2769, 1599, 3614, 2110,  739, 3322, 1690, 2110,  739,  511,  102,
            0,    0,    0,    0,    0,    0,    0,    0]])
torch.Size([1, 20])
tensor([[[nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         ...,
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan],
         [nan, nan, nan,  ..., nan, nan, nan]]], grad_fn=<LogSoftmaxBackward0>)
torch.Size([1, 20, 21128])


C:\Users\icyw\AppData\Local\Temp\ipykernel_6048\2221076008.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  source_mask1=torch.tensor(x_index!=0).unsqueeze(1).unsqueeze(1)
C:\Users\icyw\AppData\Local\Temp\ipykernel_6048\2221076008.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  source_mask2=torch.tensor(x_index!=0).unsqueeze(1).unsqueeze(1)
